# LLM-Assisted Knowledge Graph Completion untuk Pemetaan Interkoneksi Materi Sains (Fisika, Kimia, Biologi) pada Kurikulum Merdeka Tingkat SMA

## 🧱 STEP 1 — Define Ontology for SMA Science
### Classes
Class: Fase
Class: MataPelajaran
Class: Bab
Class: Subbab
Class: Konsep
Class: Subkonsep
Class: CapaianPembelajaran
Class: DomainKonsep
Class: RelasiInterdisipliner

### Object Properties
hasFase (MataPelajaran → Fase)

hasBab (MataPelajaran → Bab)
hasSubbab (Bab → Subbab)
hasKonsep (Subbab → Konsep)
hasSubkonsep (Konsep → Subkonsep)

supportsCP (Konsep → CapaianPembelajaran)

belongsToDomain (Konsep → DomainKonsep)

isRelatedTo (Konsep → Konsep)
isPrerequisiteOf (Konsep → Konsep)
explains (Konsep → Konsep)
causes (Konsep → Konsep)
appliesTo (Konsep → Konsep)
analogousTo (Konsep → Konsep)

### DomainKonsep Instances
Energi
Materi
Gaya_dan_Interaksi
Struktur
Sistem
Perubahan
Kesetimbangan
Informasi_Biologis
Skala_dan_Representasi

### Example Ontology Instantiation (Concrete Example)

Fisika

Bab: Usaha dan Energi
Konsep: Energi Kinetik
belongsToDomain → Energi

Kimia

Bab: Termokimia
Konsep: Energi Reaksi
belongsToDomain → Energi

Biologi

Bab: Metabolisme
Konsep: ATP
belongsToDomain → Energi

----------

# 🤖 STEP 2 — Large-Scale PDF Processing Pipeline (300+ pages)

Now the technical part.

## 🔹 2.1 PDF Preprocessing

Because textbook is large:

Pipeline:

1.  Extract text (PyMuPDF / pdfplumber)
    
2.  Detect:
    
    -   Bab
        
    -   Subbab
        
    -   Headings
        
3.  Chunk content intelligently:
    
    -   1 chunk ≈ 1 subbab
        
    -   NOT random token chunks
        
4.  Clean:
    
    -   Remove index pages
        
    -   Remove exercises (optional)
        
    -   Keep conceptual explanations
        

⚠️ Important:  
Structure-aware chunking > naive token chunking.

----------

## 🔹 2.2 LLM Concept Extraction

For each chunk:

Prompt structure:

-   Extract:
    
    -   Konsep utama
        
    -   Subkonsep
        
    -   Definisi
        
    -   Hubungan kausal
        
-   Classify into ontology classes
    

Example output format (JSON):

```json
{
  "mata_pelajaran": "Fisika",
  "bab": "Energi dan Usaha",
  "konsep": [
    {
      "nama": "Energi Kinetik",
      "definisi": "...",
      "subkonsep": ["massa", "kecepatan"],
      "rumus": "Ek = 1/2 mv2"
    }
  ]
}

```

----------

## 🔹 2.1 PDF Preprocessing

### Step 2.1.1 — Install Dependencies

In [1]:
# Install required libraries for PDF processing and LLM interaction
%pip install pymupdf openai tqdm python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 2.1.2 — PDF Text Extraction with Structure Detection

Using **PyMuPDF** (`fitz`) for page-level extraction because it preserves font sizes — essential for detecting heading hierarchy (Bab vs Subbab vs body text).

In [2]:
import fitz  # PyMuPDF
import re
from pathlib import Path


def extract_pages_with_fonts(pdf_path: str) -> list[dict]:
    """
    Extract every page from the PDF as a dict containing:
      - page_num: 1-based page number
      - blocks:   list of {text, size, flags, bbox} — font-aware text blocks
      - raw_text: plain concatenated text for the page
    """
    doc = fitz.open(pdf_path)
    pages = []
    for page_num, page in enumerate(doc, start=1):
        blocks = []
        for block in page.get_text("dict")["blocks"]:
            if block.get("type") != 0:      # 0 = text block
                continue
            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    text = span["text"].strip()
                    if not text:
                        continue
                    blocks.append({
                        "text":  text,
                        "size":  round(span["size"], 1),
                        "flags": span["flags"],   # bold=2^4, italic=2^1
                        "bbox":  span["bbox"],
                    })
        raw_text = "\n".join(b["text"] for b in blocks)
        pages.append({"page_num": page_num, "blocks": blocks, "raw_text": raw_text})
    doc.close()
    print(f"  Extracted {len(pages)} pages from '{Path(pdf_path).name}'")
    return pages


# ── Config: point these at your three SMA science PDF files ──────────────────
PDF_PATHS = {
    "Fisika":  "Fisika-BS-KLS-XI.pdf",
    "Kimia":   "Kimia-BS-KLS-XI.pdf",
    "Biologi": "Biologi-BS-KLS-XI.pdf",
}
GRADE = "XI"

# Extract all subjects (skip missing files gracefully)
all_subject_pages: dict[str, list[dict]] = {}
for subject, path in PDF_PATHS.items():
    if Path(path).exists():
        print(f"Processing {subject}...")
        all_subject_pages[subject] = extract_pages_with_fonts(path)
    else:
        print(f"⚠ File not found, skipping: {path}")

print(f"\n✓ Loaded subjects: {list(all_subject_pages.keys())}")

Processing Fisika...
  Extracted 248 pages from 'Fisika-BS-KLS-XI.pdf'
Processing Kimia...
  Extracted 240 pages from 'Kimia-BS-KLS-XI.pdf'
Processing Biologi...
  Extracted 312 pages from 'Biologi-BS-KLS-XI.pdf'

✓ Loaded subjects: ['Fisika', 'Kimia', 'Biologi']


### Step 2.1.3 — Heading Detection (Bab & Subbab)

Font-size heuristics + keyword patterns detect structural boundaries without relying on a rigid page-count window.

In [3]:
# Indonesian SMA textbook chapter/section heading patterns
BAB_PATTERN    = re.compile(r'^\s*(BAB\s+[IVXLCDM\d]+[:\.\s]*.{0,60})', re.IGNORECASE)
SUBBAB_PATTERN = re.compile(
    r'^\s*([A-Z]\.\s+\S.{0,80}|'          # A.  Title style
    r'\d+\.\d+\s+\S.{0,80}|'              # 1.1 Title style
    r'(?:Subbab|Sub-Bab)\s+.{0,60})',     # explicit keyword
    re.IGNORECASE
)


def classify_block(block: dict, median_size: float) -> str:
    """Return 'bab', 'subbab', or 'body' for a single text span block."""
    text  = block["text"]
    size  = block["size"]
    bold  = bool(block["flags"] & (1 << 4))  # bold flag

    if BAB_PATTERN.match(text):
        return "bab"
    if SUBBAB_PATTERN.match(text) and (size > median_size or bold):
        return "subbab"
    # Font-only heuristic (no explicit keyword)
    if size >= median_size * 1.35 and bold:
        return "bab"
    if size >= median_size * 1.15 and bold:
        return "subbab"
    return "body"


def detect_headings(pages: list[dict]) -> list[dict]:
    """
    Annotate each block with its heading level.
    Returns the same page list, each page having blocks with an added 'level' key.
    """
    # Compute median font size across the whole document for relative thresholds
    all_sizes = [b["size"] for page in pages for b in page["blocks"]]
    if not all_sizes:
        return pages
    all_sizes.sort()
    median_size = all_sizes[len(all_sizes) // 2]

    for page in pages:
        for block in page["blocks"]:
            block["level"] = classify_block(block, median_size)

    # Quick stats
    bab_count    = sum(1 for p in pages for b in p["blocks"] if b["level"] == "bab")
    subbab_count = sum(1 for p in pages for b in p["blocks"] if b["level"] == "subbab")
    print(f"  Median font size: {median_size:.1f}pt | Bab headings: {bab_count} | Subbab headings: {subbab_count}")
    return pages


# Run heading detection on all loaded subjects
for subject, pages in all_subject_pages.items():
    print(f"\nDetecting headings — {subject}:")
    all_subject_pages[subject] = detect_headings(pages)


Detecting headings — Fisika:
  Median font size: 10.0pt | Bab headings: 320 | Subbab headings: 281

Detecting headings — Kimia:
  Median font size: 10.0pt | Bab headings: 323 | Subbab headings: 294

Detecting headings — Biologi:
  Median font size: 11.0pt | Bab headings: 191 | Subbab headings: 172


### Step 2.1.4 — Structure-Aware Chunking (per Subbab)

Each chunk = one Subbab section.  
Structure: `{mata_pelajaran, bab_num, bab_title, subbab_num, subbab_title, pages[], raw_text}`

This is **content-boundary chunking** — far superior to fixed-token windows for preserving conceptual cohesion.

In [4]:
def chunk_by_structure(pages: list[dict], subject: str, grade: str = "XI") -> list[dict]:
    """
    Walk through annotated pages and split into subbab-level chunks.

    A new Bab resets both bab counter and subbab counter.
    A new Subbab (within the same Bab) produces a chunk for the previous subbab.
    Body text is accumulated into the current subbab buffer.

    Returns a list of chunk dicts ready for downstream processing.
    """
    chunks: list[dict] = []

    current_bab_num     = 0
    current_bab_title   = "Pendahuluan"
    current_subbab_num  = 0
    current_subbab_title = ""
    current_pages: list[int] = []
    current_blocks: list[str] = []

    def flush(bab_num, bab_title, subbab_num, subbab_title, pages_seen, blocks):
        """Save the accumulated buffer as a chunk."""
        raw = " ".join(blocks).strip()
        if not raw:
            return
        chunks.append({
            "mata_pelajaran":  subject,
            "grade":           grade,
            "bab_num":         bab_num,
            "bab_title":       bab_title.strip(),
            "subbab_num":      subbab_num,
            "subbab_title":    subbab_title.strip(),
            "page_range":      [min(pages_seen), max(pages_seen)] if pages_seen else [0, 0],
            "raw_text":        raw,
        })

    for page in pages:
        for block in page["blocks"]:
            level = block["level"]
            text  = block["text"].strip()
            pnum  = page["page_num"]

            if level == "bab":
                # Flush previous subbab
                flush(current_bab_num, current_bab_title,
                      current_subbab_num, current_subbab_title,
                      current_pages, current_blocks)
                current_bab_num    += 1
                current_bab_title   = text
                current_subbab_num  = 0
                current_subbab_title = ""
                current_pages       = [pnum]
                current_blocks      = []

            elif level == "subbab":
                # Flush previous subbab
                flush(current_bab_num, current_bab_title,
                      current_subbab_num, current_subbab_title,
                      current_pages, current_blocks)
                current_subbab_num  += 1
                current_subbab_title = text
                current_pages        = [pnum]
                current_blocks       = []

            else:  # body
                current_blocks.append(text)
                if pnum not in current_pages:
                    current_pages.append(pnum)

    # Flush the last open subbab
    flush(current_bab_num, current_bab_title,
          current_subbab_num, current_subbab_title,
          current_pages, current_blocks)

    return chunks


# Build chunks for every loaded subject
all_chunks: dict[str, list[dict]] = {}
for subject, pages in all_subject_pages.items():
    chunks = chunk_by_structure(pages, subject, grade=GRADE)
    all_chunks[subject] = chunks
    print(f"{subject}: {len(chunks)} subbab chunks "
          f"across {max(c['bab_num'] for c in chunks)} bab")

# Merge into a single flat list across all subjects
all_chunks_flat: list[dict] = [c for chunks in all_chunks.values() for c in chunks]
print(f"\nTotal chunks (all subjects): {len(all_chunks_flat)}")

Fisika: 509 subbab chunks across 320 bab
Kimia: 490 subbab chunks across 323 bab
Biologi: 334 subbab chunks across 191 bab

Total chunks (all subjects): 1333


### Step 2.1.5 — Text Cleaning & Normalization

Remove PDF extraction noise: page numbers, broken hyphenated words, excessive whitespace, index artefacts, and non-content headers/footers.

In [5]:
def clean_text(text: str) -> str:
    """
    Normalizes raw PDF-extracted text:
    1. Rejoin words broken with a hyphen at line boundary  (bio-\nologi → biologi)
    2. Collapse newlines / tabs into a single space
    3. Remove standalone page numbers (1–3 digit isolated numbers)
    4. Strip short all-caps header/footer artefacts (≤ 4 words)
    5. Remove non-printable / control characters
    6. Collapse multiple spaces
    """
    # 1. Hyphenated line-break rejoining
    text = re.sub(r'-\s*\n\s*', '', text)
    # 2. Newlines → space
    text = re.sub(r'[\n\t\r]+', ' ', text)
    # 3. Standalone page numbers
    text = re.sub(r'(?<!\w)\d{1,3}(?!\w)', '', text)
    # 4. Short all-caps strings (headers/footers like "BIOLOGI XI")
    text = re.sub(
        r'\b([A-Z]{2,}(\s+[A-Z]{2,}){0,3})\b',
        lambda m: '' if len(m.group().split()) <= 4 else m.group(),
        text
    )
    # 5. Non-printable chars (keep Latin extended for Indonesian diacritics)
    text = re.sub(r'[^\x20-\x7E\u00C0-\u024F\u1E00-\u1EFF]', ' ', text)
    # 6. Collapse whitespace
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()


# Apply cleaning and attach `cleaned_text` to every chunk
for chunk in all_chunks_flat:
    chunk["cleaned_text"] = clean_text(chunk["raw_text"])

# Sanity check — show before / after for the first meaningful chunk
sample = next((c for c in all_chunks_flat if len(c["raw_text"]) > 100), None)
if sample:
    print("=== RAW (first 300 chars) ===")
    print(sample["raw_text"][:300])
    print("\n=== CLEANED (first 300 chars) ===")
    print(sample["cleaned_text"][:300])
    print(f"\nSubject: {sample['mata_pelajaran']} | "
          f"Bab {sample['bab_num']}: {sample['bab_title']} | "
          f"Subbab {sample['subbab_num']}: {sample['subbab_title']}")

=== RAW (first 300 chars) ===
SMA/MA KELAS XI KEMENTERIAN PENDIDIKAN, KEBUDAYAAN, RISET, DAN TEKNOLOGI 2022 Hak Cipta pada Kementerian Pendidikan, Kebudayaan, Riset, dan Teknologi Republik Indonesia Dilindungi Undang-Undang Penaian: Buku ini disiapkan oleh Pemerintah dalam rangka pemenuhan kebutuhan buku pendidikan yang bermutu,

=== CLEANED (first 300 chars) ===
/ , , , 2022 Hak Cipta pada Kementerian Pendidikan, Kebudayaan, Riset, dan Teknologi Republik Indonesia Dilindungi Undang-Undang Penaian: Buku ini disiapkan oleh Pemerintah dalam rangka pemenuhan kebutuhan buku pendidikan yang bermutu, murah, dan merata sesuai dengan amanat dalam No. Tahun 2017. Buk

Subject: Fisika | Bab 1: FISIKA | Subbab 0: 


---
## 🔹 2.2 LLM Concept Extraction

### Step 2.2.1 — LLM Client Setup

Uses the **OpenAI-compatible** API.  
Set `OPENAI_API_KEY` in a `.env` file or as an environment variable.  
Change `MODEL_NAME` to any compatible model (GPT-4o, GPT-4-turbo, etc.).

In [6]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()  # loads OPENAI_API_KEY from .env if present

MODEL_NAME = os.getenv("LLM_MODEL", "gpt-4o-mini")   # override via env var
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Quick connectivity test
try:
    test = client.models.list()
    print(f"✓ OpenAI API reachable | Using model: {MODEL_NAME}")
except Exception as e:
    print(f"⚠ API connection issue: {e}\n  Set OPENAI_API_KEY in your .env file.")

✓ OpenAI API reachable | Using model: gpt-4o-mini


### Step 2.2.2 — Prompt Engineering for Concept Extraction

The system prompt enforces strict JSON output aligned with the ontology defined in Step 1.  
Each extraction captures: `konsep`, `subkonsep`, `definisi`, `rumus`, `domain_konsep`, and inter-concept relations.

In [7]:
SYSTEM_PROMPT = """\
Kamu adalah asisten ekstraksi pengetahuan ilmiah untuk buku teks SMA Kurikulum Merdeka Indonesia.
Tugasmu: membaca bagian teks pelajaran dan mengekstrak konsep-konsep ilmiah beserta relasinya
sesuai ontologi yang telah didefinisikan.

Ontologi domain konsep yang tersedia:
  Energi, Materi, Gaya_dan_Interaksi, Struktur, Sistem,
  Perubahan, Kesetimbangan, Informasi_Biologis, Skala_dan_Representasi

Tipe relasi antar konsep yang diizinkan:
  isRelatedTo, isPrerequisiteOf, explains, causes, appliesTo, analogousTo

Kembalikan HANYA satu objek JSON yang valid dengan struktur berikut (tanpa teks tambahan):
{
  "mata_pelajaran": "<Fisika|Kimia|Biologi>",
  "bab": "<judul bab>",
  "subbab": "<judul subbab>",
  "konsep": [
    {
      "nama": "<nama konsep utama>",
      "definisi": "<definisi singkat dalam 1-2 kalimat>",
      "subkonsep": ["<subkonsep 1>", "..."],
      "rumus": "<rumus atau simbol utama, kosongkan jika tidak ada>",
      "domain_konsep": "<salah satu dari daftar domain di atas>",
      "relasi": [
        {
          "tipe": "<tipe relasi>",
          "target": "<nama konsep tujuan>",
          "keterangan": "<penjelasan singkat mengapa relasi ini ada>"
        }
      ]
    }
  ]
}
"""

MAX_CHUNK_CHARS = 3000   # Truncate very long subbab texts to stay inside context window


def build_user_prompt(chunk: dict) -> str:
    """Build the per-chunk user message."""
    text = chunk["cleaned_text"][:MAX_CHUNK_CHARS]
    return (
        f"Mata Pelajaran: {chunk['mata_pelajaran']}\n"
        f"Bab: {chunk['bab_title']}\n"
        f"Subbab: {chunk['subbab_title']}\n\n"
        f"Teks:\n{text}"
    )


# Preview prompt for the first non-empty chunk
sample_chunk = next((c for c in all_chunks_flat if len(c.get("cleaned_text", "")) > 50), None)
if sample_chunk:
    print("=== SYSTEM PROMPT (first 300 chars) ===")
    print(SYSTEM_PROMPT[:300], "...")
    print("\n=== USER PROMPT (first 300 chars) ===")
    print(build_user_prompt(sample_chunk)[:300], "...")

=== SYSTEM PROMPT (first 300 chars) ===
Kamu adalah asisten ekstraksi pengetahuan ilmiah untuk buku teks SMA Kurikulum Merdeka Indonesia.
Tugasmu: membaca bagian teks pelajaran dan mengekstrak konsep-konsep ilmiah beserta relasinya
sesuai ontologi yang telah didefinisikan.

Ontologi domain konsep yang tersedia:
  Energi, Materi, Gaya_dan_ ...

=== USER PROMPT (first 300 chars) ===
Mata Pelajaran: Fisika
Bab: FISIKA
Subbab: 

Teks:
/ , , , 2022 Hak Cipta pada Kementerian Pendidikan, Kebudayaan, Riset, dan Teknologi Republik Indonesia Dilindungi Undang-Undang Penaian: Buku ini disiapkan oleh Pemerintah dalam rangka pemenuhan kebutuhan buku pendidikan yang bermutu, murah, dan me ...


### Step 2.2.3 — Batch LLM Concept Extraction

Iterates over all subbab chunks and calls the LLM for each.  
- Retries on transient API errors (up to 3× with exponential back-off)  
- Skips chunks with insufficient text  
- Stores raw LLM response + parsed JSON on each chunk object

⚠ **Cost estimate**: ~300 chunks × ~800 tokens ≈ 240 k tokens total.  
Use `LIMIT_CHUNKS` below to process a subset during testing.

In [8]:
import time
from tqdm.auto import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
LIMIT_CHUNKS   = None   # Set to e.g. 10 to test on the first 10 chunks only
MIN_TEXT_LEN   = 80     # Skip subbab chunks shorter than this (likely navigation pages)
MAX_RETRIES    = 3
RETRY_DELAY    = 5      # seconds (doubles on each retry)
# ──────────────────────────────────────────────────────────────────────────────

def call_llm(chunk: dict) -> dict | None:
    """
    Send one chunk to the LLM and return the parsed JSON extraction.
    Returns None on persistent failure.
    """
    user_msg = build_user_prompt(chunk)
    delay = RETRY_DELAY

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_msg},
                ],
                temperature=0.0,
                response_format={"type": "json_object"},
            )
            raw_json = response.choices[0].message.content.strip()
            return json.loads(raw_json)

        except json.JSONDecodeError as e:
            print(f"  [JSON parse error attempt {attempt}]: {e}")
            return None   # No point retrying a deterministic parse failure

        except Exception as e:
            if attempt == MAX_RETRIES:
                print(f"  [API error, giving up]: {e}")
                return None
            print(f"  [API error attempt {attempt}, retrying in {delay}s]: {e}")
            time.sleep(delay)
            delay *= 2

    return None


# Run extraction pipeline
target_chunks = all_chunks_flat
if LIMIT_CHUNKS:
    target_chunks = [c for c in all_chunks_flat if len(c.get("cleaned_text", "")) >= MIN_TEXT_LEN][:LIMIT_CHUNKS]
else:
    target_chunks = [c for c in all_chunks_flat if len(c.get("cleaned_text", "")) >= MIN_TEXT_LEN]

extracted_concepts: list[dict] = []
failed_chunks:      list[dict] = []

print(f"Processing {len(target_chunks)} chunks with {MODEL_NAME}...\n")

for chunk in tqdm(target_chunks, desc="LLM extraction"):
    result = call_llm(chunk)
    if result:
        # Attach provenance metadata to the extracted result
        result["_source"] = {
            "mata_pelajaran": chunk["mata_pelajaran"],
            "grade":          chunk["grade"],
            "bab_num":        chunk["bab_num"],
            "bab_title":      chunk["bab_title"],
            "subbab_num":     chunk["subbab_num"],
            "subbab_title":   chunk["subbab_title"],
            "page_range":     chunk["page_range"],
        }
        extracted_concepts.append(result)
    else:
        failed_chunks.append(chunk)

print(f"\n✓ Successfully extracted: {len(extracted_concepts)} chunks")
print(f"✗ Failed / skipped:      {len(failed_chunks)} chunks")

# Preview first extraction result
if extracted_concepts:
    print("\n=== Sample Extraction ===")
    sample = extracted_concepts[0]
    print(f"Subject: {sample.get('mata_pelajaran')} | Bab: {sample.get('bab')} | Subbab: {sample.get('subbab')}")
    for k in sample.get("konsep", [])[:2]:
        print(f"  Konsep : {k.get('nama')}")
        print(f"  Domain : {k.get('domain_konsep')}")
        print(f"  Definisi: {k.get('definisi', '')[:100]}")
        print(f"  Relasi : {k.get('relasi', [])[:2]}")
        print()

Processing 1041 chunks with gpt-4o-mini...



LLM extraction:   0%|          | 0/1041 [00:00<?, ?it/s]

  [API error attempt 1, retrying in 5s]: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
  [API error attempt 2, retrying in 10s]: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
  [API error, giving up]: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'par

KeyboardInterrupt: 

### Step 2.2.4 — Persist Extraction Results

Saves three artefacts:
- `extracted_concepts.json` — full per-subbab extraction (input to Step 3 KG construction)  
- `extraction_summary.json` — lightweight stats (no concept bodies) for quick inspection  
- `failed_chunks.json` — chunks that failed LLM parsing (for manual review or re-run)

In [ ]:
from collections import Counter

# ── 1. Full extraction results ────────────────────────────────────────────────
OUTPUT_PATH = "extracted_concepts.json"
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(extracted_concepts, f, ensure_ascii=False, indent=2)
print(f"✓ Saved full extractions  → {OUTPUT_PATH}  ({len(extracted_concepts)} entries)")

# ── 2. Summary (no concept bodies) ───────────────────────────────────────────
summary_records = []
for entry in extracted_concepts:
    src = entry.get("_source", {})
    summary_records.append({
        "mata_pelajaran": src.get("mata_pelajaran"),
        "grade":          src.get("grade"),
        "bab_num":        src.get("bab_num"),
        "bab_title":      src.get("bab_title"),
        "subbab_num":     src.get("subbab_num"),
        "subbab_title":   src.get("subbab_title"),
        "page_range":     src.get("page_range"),
        "num_konsep":     len(entry.get("konsep", [])),
    })

SUMMARY_PATH = "extraction_summary.json"
with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary_records, f, ensure_ascii=False, indent=2)
print(f"✓ Saved extraction summary → {SUMMARY_PATH}")

# ── 3. Failed chunks ──────────────────────────────────────────────────────────
if failed_chunks:
    FAILED_PATH = "failed_chunks.json"
    with open(FAILED_PATH, "w", encoding="utf-8") as f:
        json.dump(failed_chunks, f, ensure_ascii=False, indent=2)
    print(f"✓ Saved failed chunks      → {FAILED_PATH}  ({len(failed_chunks)} entries)")

# ── 4. Pipeline statistics ────────────────────────────────────────────────────
print("\n══ Pipeline Statistics ══")
total_konsep  = sum(len(e.get("konsep", [])) for e in extracted_concepts)
domain_counts = Counter(
    k.get("domain_konsep", "Unknown")
    for e in extracted_concepts
    for k in e.get("konsep", [])
)
subject_counts = Counter(e.get("mata_pelajaran") for e in extracted_concepts)

print(f"  Total subbab chunks processed : {len(extracted_concepts) + len(failed_chunks)}")
print(f"  Successfully extracted         : {len(extracted_concepts)}")
print(f"  Total konsep extracted         : {total_konsep}")
print(f"\n  Konsep per subject:")
for subj, cnt in subject_counts.most_common():
    subj_konsep = sum(len(e.get("konsep",[])) for e in extracted_concepts if e.get("mata_pelajaran")==subj)
    print(f"    {subj:10s} → {cnt} subbab | {subj_konsep} konsep")

print(f"\n  Konsep per DomainKonsep:")
for domain, cnt in domain_counts.most_common():
    print(f"    {domain:30s} : {cnt}")